
# Phase 6 — Reproducibility and Robustness Verification

## Research Objective

Evaluate whether the SHML system produces reproducible
experimental results and handles unsafe recovery attempts
according to predefined safety criteria.

## Research Questions

1. Can the Phase 5 experiment be reproduced using the
   same dataset, split configuration, and random seed?

2. Do repeated executions produce consistent recovery
   metrics and acceptance decisions?

3. Does the safety controller reject deliberately
   unsafe recovery candidates?

4. Does the fallback mechanism preserve the previously
   approved pipeline state when recovery fails?

## Scope

- Dataset: UCI Default of Credit Card Clients
- Model: Logistic Regression
- Monitored feature: BILL_AMT1
- Failure type: Injected missingness
- Recovery: Training-median imputation
- Missingness levels: 0%, 5%, 15%, 30%
- Random state: 42
- Primary metric: F1-score

This phase focuses on reproducibility and safety verification
without introducing additional failure types or models.


In [8]:
import numpy as np 
import pandas as pd
from pathlib import Path 

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Phase 6 libraries imported successfully.")
print("Random state:", RANDOM_STATE)

Phase 6 libraries imported successfully.
Random state: 42


In [9]:
performance_df = pd.read_csv(performance_path)
safety_df = pd.read_csv(safety_path)
summary_df = pd.read_csv(summary_path)

In [10]:

# Phase 6.1 — Load Phase 5 artifacts

results_dir = Path("results/phase5")

performance_path = (
    results_dir / "phase5_recovery_performance.csv"
)

safety_path = (
    results_dir / "phase5_safety_decisions.csv"
)

summary_path = (
    results_dir / "phase5_final_experiment_summary.csv"
)

# Verify that required artifacts exist
for path in [
    performance_path,
    safety_path,
    summary_path
]:
    assert path.exists(), f"Missing artifact: {path}"

# Load saved experiment results
performance_df = pd.read_csv(performance_path)
safety_df = pd.read_csv(safety_path)
summary_df = pd.read_csv(summary_path)

print("Phase 5 artifacts loaded successfully.")

display(summary_df.round(4))


Phase 5 artifacts loaded successfully.


,Missingness,Accuracy,Precision,Recall,F1,ROC-AUC,F1 Loss (%),Recall Loss (%),Data Integrity,Decision,Fallback Action,Batch Blocked,Manual Review
0,0.00,0.8087,0.6948,0.2402,0.3570,0.7130,0.0000,0.0000,True,ACCEPT,APPROVE_FOR_OFFLINE_EVALUATION,False,False
1,0.05,0.8087,0.6925,0.2422,0.3589,0.7132,-0.5365,-0.8368,True,ACCEPT,APPROVE_FOR_OFFLINE_EVALUATION,False,False
2,0.15,0.8080,0.6835,0.2452,0.3609,0.7090,-1.1104,-2.0921,True,ACCEPT,APPROVE_FOR_OFFLINE_EVALUATION,False,False
3,0.30,0.8093,0.6827,0.2573,0.3737,0.7111,-4.6892,-7.1130,True,ACCEPT,APPROVE_FOR_OFFLINE_EVALUATION,False,False


In [11]:

# Phase 6.1 — Verify saved experiment structure

expected_levels = [0.00, 0.05, 0.15, 0.30]

required_columns = [
    "Missingness",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "Decision"
]

# Check required columns
assert all(
    col in summary_df.columns
    for col in required_columns
), "Required experiment columns are missing."

# Check experiment count
assert len(summary_df) == 4, (
    "Expected exactly four experiment scenarios."
)

# Check missingness levels
np.testing.assert_allclose(
    sorted(summary_df["Missingness"].tolist()),
    expected_levels,
    atol=1e-10
)

# Check for missing evaluation values
assert summary_df[required_columns].notna().all().all()

# Check valid metric ranges
metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

assert summary_df[metric_columns].ge(0).all().all()
assert summary_df[metric_columns].le(1).all().all()

# Check valid decisions
assert summary_df["Decision"].isin(
    ["ACCEPT", "REJECT"]
).all()

print("Phase 6.1 verification passed.")
print("All saved experiment artifacts are structurally valid.")


Phase 6.1 verification passed.
All saved experiment artifacts are structurally valid.



## Step 6.2 — Independent Experiment Reproduction

### Objective

Reproduce the Phase 5 missingness experiment using the
same dataset, training configuration, random seed,
feature selection, and recovery strategy.

### Experimental Controls

- Dataset: UCI Default of Credit Card Clients
- Model: Logistic Regression
- Random state: 42
- Training split: 70%
- Validation split: 15%
- Test split: 15%
- Failure feature: BILL_AMT1
- Missingness: 0%, 5%, 15%, 30%
- Recovery: Training-derived median imputation

### Reproducibility Criteria

1. Recreate the original train/validation/test splits.
2. Refit the original model and scaler.
3. Regenerate the missingness scenarios.
4. Apply the original recovery method.
5. Recalculate performance metrics.
6. Compare results against saved Phase 5 artifacts.

The original Phase 5 results will not be modified.


In [12]:

# Step 6.2 — Inspect saved reference results

reference_df = summary_df[
    [
        "Missingness",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "Decision"
    ]
].copy()

reference_df = reference_df.sort_values(
    "Missingness"
).reset_index(drop=True)

print("Phase 5 reference results:")
display(reference_df.round(6))

print("\nReference scenarios:", len(reference_df))
print("Random state to reproduce:", RANDOM_STATE)


Phase 5 reference results:


,Missingness,Accuracy,Precision,Recall,F1,ROC-AUC,Decision
0,0.00,0.808667,0.694767,0.240201,0.356983,0.713025,ACCEPT
1,0.05,0.808667,0.692529,0.242211,0.358898,0.713210,ACCEPT
2,0.15,0.808000,0.683473,0.245226,0.360947,0.708958,ACCEPT
3,0.30,0.809333,0.682667,0.257286,0.373723,0.711135,ACCEPT



Reference scenarios: 4
Random state to reproduce: 42


In [13]:
# Phase 6.2 — Independent reproduction

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [15]:
# 1. Load the original dataset
data_dir = Path("data/working")
candidates = sorted(
    data_dir.glob("*credit*card*clients*.xls*")
)

if not candidates:
    raise FileNotFoundError(
        f"Dataset not found in {data_dir}"
    )

df_repro = pd.read_excel(candidates[0], header=1)
df_repro.head()



,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [16]:
target = "default payment next month"
X = df_repro.drop(columns=["ID", target])
y = df_repro[target]

# 2. Reproduce the original stratified splits

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# 3. Refit scaler and logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_scaled, y_train)

# 4. Recreate the failure and recovery experiments
reproduction_results = []

training_median = X_train["BILL_AMT1"].median()

for rate in [0.00, 0.05, 0.15, 0.30]:

    X_corrupted = X_val.copy(deep=True)

    rng = np.random.default_rng(42)

    n_missing = int(len(X_corrupted) * rate)

    positions = rng.choice(
        len(X_corrupted),
        size=n_missing,
        replace=False
    )

    feature_idx = X_corrupted.columns.get_loc(
        "BILL_AMT1"
    )

    X_corrupted.iloc[positions, feature_idx] = np.nan

    # Apply original recovery strategy
    X_recovered = X_corrupted.copy(deep=True)

    X_recovered["BILL_AMT1"] = (
        X_recovered["BILL_AMT1"].fillna(training_median)
    )

    # Evaluate with unchanged model and scaler
    X_scaled = scaler.transform(X_recovered)

    predictions = model.predict(X_scaled)
    probabilities = model.predict_proba(X_scaled)[:, 1]

    reproduction_results.append({
        "Missingness": rate,
        "Accuracy": accuracy_score(y_val, predictions),
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_val, probabilities)
    })

reproduced_df = pd.DataFrame(reproduction_results)

print("Phase 6.2 — Reproduced Experiment Results")
display(reproduced_df.round(6))


Phase 6.2 — Reproduced Experiment Results


,Missingness,Accuracy,Precision,Recall,F1,ROC-AUC
0,0.00,0.808667,0.694767,0.240201,0.356983,0.713025
1,0.05,0.808667,0.692529,0.242211,0.358898,0.713210
2,0.15,0.808000,0.683473,0.245226,0.360947,0.708958
3,0.30,0.809333,0.682667,0.257286,0.373723,0.711135


In [17]:

# Phase 6.3 — Reproducibility comparison

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

# Sort both tables by missingness
original = reference_df.sort_values(
    "Missingness"
).reset_index(drop=True)

reproduced = reproduced_df.sort_values(
    "Missingness"
).reset_index(drop=True)

# Verify matching scenarios
np.testing.assert_allclose(
    original["Missingness"],
    reproduced["Missingness"],
    atol=1e-12
)

# Calculate absolute differences
comparison_df = pd.DataFrame({
    "Missingness": original["Missingness"],
    "Original F1": original["F1"],
    "Reproduced F1": reproduced["F1"],
    "F1 Difference": abs(
        original["F1"] - reproduced["F1"]
    ),
    "Original ROC-AUC": original["ROC-AUC"],
    "Reproduced ROC-AUC": reproduced["ROC-AUC"]
})

display(comparison_df.round(8))

# Compare all five evaluation metrics
np.testing.assert_allclose(
    original[metric_columns].to_numpy(),
    reproduced[metric_columns].to_numpy(),
    rtol=1e-6,
    atol=1e-8
)

print("Phase 6.3 verification passed.")
print("All reproduced metrics match Phase 5.")


,Missingness,Original F1,Reproduced F1,F1 Difference,Original ROC-AUC,Reproduced ROC-AUC
0,0.00,0.356983,0.356983,0.0,0.713025,0.713025
1,0.05,0.358898,0.358898,0.0,0.713210,0.713210
2,0.15,0.360947,0.360947,0.0,0.708958,0.708958
3,0.30,0.373723,0.373723,0.0,0.711135,0.711135


Phase 6.3 verification passed.
All reproduced metrics match Phase 5.



## Step 6.4 — Unsafe Recovery and Rejection Testing

### Objective

Verify that the SHML safety controller rejects recovery
candidates that violate predefined performance or
data-integrity requirements.

### Test Cases

1. Excessive F1 degradation.
2. Excessive recall degradation.
3. Failed data-integrity validation.
4. Valid recovery candidate (control test).

### Acceptance Criteria

The controller must reject every unsafe candidate and
accept a valid candidate that satisfies all safety checks.

These tests evaluate decision logic rather than
predictive model performance.


In [18]:

# Phase 6.4 — Safety controller verification

def verify_recovery_safety(
    recovered_metrics,
    healthy_metrics,
    data_integrity_passed,
    max_f1_loss=0.10,
    max_recall_loss=0.10
):
    required = ["f1", "recall"]

    if any(
        key not in recovered_metrics
        or not np.isfinite(recovered_metrics[key])
        or not 0 <= recovered_metrics[key] <= 1
        for key in required
    ):
        return "REJECT"

    if (
        healthy_metrics["f1"] <= 0
        or healthy_metrics["recall"] <= 0
    ):
        return "REJECT"

    f1_loss = (
        healthy_metrics["f1"] - recovered_metrics["f1"]
    ) / healthy_metrics["f1"]

    recall_loss = (
        healthy_metrics["recall"] - recovered_metrics["recall"]
    ) / healthy_metrics["recall"]

    if not data_integrity_passed:
        return "REJECT"

    if f1_loss > max_f1_loss:
        return "REJECT"

    if recall_loss > max_recall_loss:
        return "REJECT"

    return "ACCEPT"


# Use reproduced healthy metrics
healthy_row = reproduced_df.loc[
    reproduced_df["Missingness"] == 0.0
].iloc[0]

healthy_reference = {
    "f1": float(healthy_row["F1"]),
    "recall": float(healthy_row["Recall"])
}

test_cases = [
    {
        "Test": "Excessive F1 loss",
        "f1_factor": 0.75,
        "recall_factor": 1.00,
        "integrity": True,
        "expected": "REJECT"
    },
    {
        "Test": "Excessive recall loss",
        "f1_factor": 1.00,
        "recall_factor": 0.75,
        "integrity": True,
        "expected": "REJECT"
    },
    {
        "Test": "Data integrity failure",
        "f1_factor": 1.00,
        "recall_factor": 1.00,
        "integrity": False,
        "expected": "REJECT"
    },
    {
        "Test": "Valid recovery",
        "f1_factor": 0.95,
        "recall_factor": 0.95,
        "integrity": True,
        "expected": "ACCEPT"
    }
]

safety_test_results = []

for case in test_cases:
    candidate = {
        "f1": healthy_reference["f1"] * case["f1_factor"],
        "recall": healthy_reference["recall"] * case["recall_factor"]
    }

    actual = verify_recovery_safety(
        candidate,
        healthy_reference,
        case["integrity"]
    )

    safety_test_results.append({
        "Test": case["Test"],
        "Expected": case["expected"],
        "Actual": actual,
        "Passed": actual == case["expected"]
    })

safety_test_df = pd.DataFrame(safety_test_results)

display(safety_test_df)

assert safety_test_df["Passed"].all()

print("Phase 6.4 verification passed.")
print("All safety controller tests passed.")


,Test,Expected,Actual,Passed
0,Excessive F1 loss,REJECT,REJECT,True
1,Excessive recall loss,REJECT,REJECT,True
2,Data integrity failure,REJECT,REJECT,True
3,Valid recovery,ACCEPT,ACCEPT,True


Phase 6.4 verification passed.
All safety controller tests passed.



## Step 6.5 — Fallback and State Preservation

### Objective

Verify that rejected recovery candidates do not modify
the previously approved SHML pipeline state.

### Safety Principle

A recovery candidate must pass all predefined safety
checks before it can replace an approved state.

### Expected Behavior

- ACCEPT: Candidate may become the approved state.
- REJECT: Preserve the previous approved state.
- REJECT: Block the unsafe candidate from promotion.
- Record every attempted state transition.

### Scope

This experiment simulates deployment-state management.
It does not modify a production model or deploy to cloud.


In [19]:

# Phase 6.5 — Simulated state preservation

from copy import deepcopy

# Initial approved pipeline state
approved_state = {
    "version": "baseline_v1",
    "model": "LogisticRegression",
    "status": "APPROVED",
    "f1": healthy_reference["f1"],
    "recall": healthy_reference["recall"]
}

initial_state = deepcopy(approved_state)

def evaluate_state_transition(
    current_state,
    candidate_version,
    candidate_metrics,
    integrity_passed
):
    # Evaluate candidate using Phase 6.4 safety rules
    decision = verify_recovery_safety(
        recovered_metrics=candidate_metrics,
        healthy_metrics=healthy_reference,
        data_integrity_passed=integrity_passed
    )

    new_state = deepcopy(current_state)

    if decision == "ACCEPT":
        new_state["version"] = candidate_version
        new_state["f1"] = candidate_metrics["f1"]
        new_state["recall"] = candidate_metrics["recall"]

        action = "PROMOTE_CANDIDATE"

    else:
        action = "PRESERVE_APPROVED_STATE"

    return {
        "decision": decision,
        "action": action,
        "state": new_state
    }


# Test an unsafe candidate
unsafe_candidate = {
    "f1": healthy_reference["f1"] * 0.70,
    "recall": healthy_reference["recall"] * 0.70
}

result = evaluate_state_transition(
    current_state=approved_state,
    candidate_version="candidate_unsafe_v2",
    candidate_metrics=unsafe_candidate,
    integrity_passed=True
)

print("Decision:", result["decision"])
print("Action:", result["action"])
print("Active version:", result["state"]["version"])

# Verify rejection and state preservation
assert result["decision"] == "REJECT"
assert result["action"] == "PRESERVE_APPROVED_STATE"
assert result["state"] == initial_state
assert approved_state == initial_state

print("Phase 6.5 rejection verification passed.")


Decision: REJECT
Action: PRESERVE_APPROVED_STATE
Active version: baseline_v1
Phase 6.5 rejection verification passed.


In [20]:

# Phase 6.5 — Safe candidate promotion test

safe_candidate = {
    "f1": healthy_reference["f1"] * 0.98,
    "recall": healthy_reference["recall"] * 0.98
}

promotion_result = evaluate_state_transition(
    current_state=approved_state,
    candidate_version="candidate_safe_v2",
    candidate_metrics=safe_candidate,
    integrity_passed=True
)

print("Decision:", promotion_result["decision"])
print("Action:", promotion_result["action"])
print("Active version:", promotion_result["state"]["version"])

# Verify candidate promotion
assert promotion_result["decision"] == "ACCEPT"
assert promotion_result["action"] == "PROMOTE_CANDIDATE"
assert promotion_result["state"]["version"] == "candidate_safe_v2"

# Verify the original input state was not mutated
assert approved_state == initial_state

print("Phase 6.5 safe promotion verification passed.")


Decision: ACCEPT
Action: PROMOTE_CANDIDATE
Active version: candidate_safe_v2
Phase 6.5 safe promotion verification passed.



## Step 6.6 — Experiment Artifacts and Reproducibility Logs

### Objective

Preserve the results of Phase 6 reproducibility,
safety-controller, and state-transition verification.

### Outputs

1. phase6_reproducibility_comparison.csv
2. phase6_safety_test_results.csv
3. phase6_state_transition_logs.csv

### Reproducibility

All generated artifacts are saved separately from Phase 5
to preserve the original experimental evidence.

### Limitations

The reproducibility test uses the same dataset, random
seed, model configuration, and execution environment.

Safety and state-transition tests use controlled synthetic
inputs and simulated deployment states.


In [21]:

# Phase 6.6 — Save experiment artifacts

from pathlib import Path

output_dir = Path("results/phase6")
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Reproducibility comparison
comparison_df.to_csv(
    output_dir / "phase6_reproducibility_comparison.csv",
    index=False
)

# 2. Safety controller test results
safety_test_df.to_csv(
    output_dir / "phase6_safety_test_results.csv",
    index=False
)

# 3. State transition verification logs
transition_logs = pd.DataFrame([
    {
        "Candidate": "candidate_unsafe_v2",
        "Decision": result["decision"],
        "Action": result["action"],
        "Active Version": result["state"]["version"],
        "Expected Decision": "REJECT",
        "Passed": (
            result["decision"] == "REJECT"
            and result["state"] == initial_state
        )
    },
    {
        "Candidate": "candidate_safe_v2",
        "Decision": promotion_result["decision"],
        "Action": promotion_result["action"],
        "Active Version": promotion_result["state"]["version"],
        "Expected Decision": "ACCEPT",
        "Passed": (
            promotion_result["decision"] == "ACCEPT"
            and promotion_result["state"]["version"]
            == "candidate_safe_v2"
        )
    }
])

transition_logs.to_csv(
    output_dir / "phase6_state_transition_logs.csv",
    index=False
)

print("Phase 6 experiment artifacts saved successfully.")

for path in sorted(output_dir.glob("*.csv")):
    print(path)


Phase 6 experiment artifacts saved successfully.
results/phase6/phase6_reproducibility_comparison.csv
results/phase6/phase6_safety_test_results.csv
results/phase6/phase6_state_transition_logs.csv


In [22]:

# Phase 6.6 — Verify saved experiment artifacts

expected_files = [
    "phase6_reproducibility_comparison.csv",
    "phase6_safety_test_results.csv",
    "phase6_state_transition_logs.csv"
]

for filename in expected_files:
    path = output_dir / filename

    assert path.exists(), f"Missing file: {filename}"

    saved_df = pd.read_csv(path)

    assert not saved_df.empty, (
        f"Empty experiment artifact: {filename}"
    )

    print(f"Verified: {filename} ({len(saved_df)} rows)")

# Confirm the verification logs report success
assert pd.read_csv(
    output_dir / "phase6_safety_test_results.csv"
)["Passed"].all()

assert pd.read_csv(
    output_dir / "phase6_state_transition_logs.csv"
)["Passed"].all()

print("\nPhase 6.6 verification passed.")


Verified: phase6_reproducibility_comparison.csv (4 rows)
Verified: phase6_safety_test_results.csv (4 rows)
Verified: phase6_state_transition_logs.csv (2 rows)

Phase 6.6 verification passed.


# Phase 6 — Reproducibility and Robustness Verification

## 1. Research Objective

Phase 6 evaluates the reproducibility of the Self-Healing Machine Learning (SHML) experiment and verifies the behavior of its recovery safety mechanisms.

The primary objectives are to:

1. Reproduce Phase 5 experimental results independently.
2. Verify consistency of predictive performance metrics.
3. Test rejection of unsafe recovery candidates.
4. Verify preservation of approved pipeline states.
5. Document reproducibility evidence and limitations.

## 2. Experimental Configuration

| Component | Configuration |
|---|---|
| Dataset | UCI Default of Credit Card Clients |
| Prediction task | Binary credit-default classification |
| Model | Logistic Regression |
| Random state | 42 |
| Training/validation/test split | 70% / 15% / 15% |
| Monitored feature | BILL_AMT1 |
| Failure mechanism | Artificially injected missingness |
| Missingness levels | 0%, 5%, 15%, 30% |
| Recovery method | Training-derived median imputation |
| Primary evaluation metric | F1-score |
| Safety thresholds | Maximum 10% relative F1 and recall degradation |

## 3. Reproducibility Experiment

The Phase 5 experiment was independently re-executed using the original dataset, stratified splitting procedure, preprocessing configuration, Logistic Regression model, random seed, and missingness injection mechanism.

The reproduced metrics were compared against the saved Phase 5 experiment results using NumPy's `assert_allclose()` with a relative tolerance of 1e-6 and absolute tolerance of 1e-8.

### Results

| Missingness | Reference F1 | Reproduced F1 |
|---|---:|---:|
| 0% | 0.356983 | 0.356983 |
| 5% | 0.358898 | 0.358898 |
| 15% | 0.360947 | 0.360947 |
| 30% | 0.373723 | 0.373723 |

All five evaluation metrics—accuracy, precision, recall, F1-score, and ROC-AUC—matched the saved Phase 5 results within the specified numerical tolerances.

**Finding:** The experiment demonstrated same-seed, same-configuration reproducibility in the tested environment.

This finding does not establish reproducibility across different environments, library versions, or random seeds.

## 4. Safety Controller Verification

Four controlled test cases were evaluated:

| Test | Expected | Actual |
|---|---|---|
| Excessive F1 degradation |